In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import pandas as pd
from src.model import TextCNN
from src.train import train_model
from src.evaluate import evaluate_model
from src.preprocess import clean_text
from src.dataset import build_vocabulary, load_glove, create_dataloaders

# ── Cargar datos con title y text ────────────────────────────────────────
df_raw = pd.read_csv("WELFake_clean.csv").dropna(subset=['text', 'label'])
df_raw['title'] = df_raw['title'].fillna('')

from sklearn.model_selection import train_test_split

# Split estratificado 80/10/10
train_df, temp_df = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['label'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── Preparar los tres campos de entrada ──────────────────────────────────
def preparar_textos(df, campo):
    if campo == "title":
        return [clean_text(t) for t in df['title']]
    elif campo == "text":
        return [clean_text(t) for t in df['text']]
    else:  # title+text
        return [clean_text(str(ti) + " " + str(tx))
                for ti, tx in zip(df['title'], df['text'])]

labels_train = train_df['label'].tolist()
labels_val   = val_df['label'].tolist()
labels_test  = test_df['label'].tolist()

# ── Vocabulario y GloVe (base: title+text del train) ────────────────────
X_train_base = preparar_textos(train_df, "title+text")
word2idx = build_vocabulary(X_train_base, max_vocab=20000)
embedding_matrix = load_glove("glove.6B.50d.txt", word2idx, embed_dim=50)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Función run_experiment ───────────────────────────────────────────────
def run_experiment(config, experiment_id):
    torch.manual_seed(42)

    # Preparar datos según campo de entrada
    X_tr  = preparar_textos(train_df, config["input_field"])
    X_val = preparar_textos(val_df,   config["input_field"])
    X_te  = preparar_textos(test_df,  config["input_field"])

    train_loader, val_loader, test_loader = create_dataloaders(
        train_data=(X_tr,  labels_train),
        val_data  =(X_val, labels_val),
        test_data =(X_te,  labels_test),
        word2idx=word2idx,
        batch_size=64,
        max_len=200
    )

    model = TextCNN(
        vocab_size=len(word2idx),
        embed_dim=50,
        num_filters=config["num_filters"],
        kernel_sizes=[3, 4, 5],
        dropout=config["dropout"],
        pretrained_embeddings=embedding_matrix
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.BCEWithLogitsLoss()

    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        num_epochs=10,
        patience=3
    )

    val_metrics  = evaluate_model(model, val_loader,  criterion, device)
    test_metrics = evaluate_model(model, test_loader, criterion, device)

    result = {
        "experiment_id": experiment_id,
        "lr":            config["lr"],
        "num_filters":   config["num_filters"],
        "dropout":       config["dropout"],
        "input_field":   config["input_field"],
        "val_f1":        round(val_metrics["f1"], 4),
        "val_accuracy":  round(val_metrics["accuracy"], 4),
        "test_f1":       round(test_metrics["f1"], 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
    }

    print(f"\n[{experiment_id}] val_f1={result['val_f1']} | test_f1={result['test_f1']}")
    return result

print("✅ Setup listo")

Train: 50488 | Val: 6311 | Test: 6311
GloVe cargado: 18,185 / 20,002 palabras encontradas (90.9% de cobertura)
✅ Setup listo


In [3]:
all_results = []

# Config base: lr=1e-3, num_filters=100, dropout=0.5, input_field=title+text

# ── Sweep 1: Learning Rate ───────────────────────────────────────────────
print("\n" + "="*50)
print("SWEEP 1: Learning Rate")
print("="*50)

for lr in [1e-2, 1e-3, 1e-4]:
    config = {"lr": lr, "num_filters": 100, "dropout": 0.5, "input_field": "title+text"}
    result = run_experiment(config, f"lr_{lr}")
    all_results.append(result)

# ── Sweep 2: Num Filters ─────────────────────────────────────────────────
print("\n" + "="*50)
print("SWEEP 2: Num Filters")
print("="*50)

for nf in [50, 100, 200]:
    if nf == 100:  # ya lo tenemos del sweep anterior
        continue
    config = {"lr": 1e-3, "num_filters": nf, "dropout": 0.5, "input_field": "title+text"}
    result = run_experiment(config, f"nf_{nf}")
    all_results.append(result)

# ── Sweep 3: Dropout ─────────────────────────────────────────────────────
print("\n" + "="*50)
print("SWEEP 3: Dropout")
print("="*50)

for dp in [0.3, 0.5, 0.7]:
    if dp == 0.5:  # ya lo tenemos
        continue
    config = {"lr": 1e-3, "num_filters": 100, "dropout": dp, "input_field": "title+text"}
    result = run_experiment(config, f"dp_{dp}")
    all_results.append(result)

# ── Sweep 4: Input Field ─────────────────────────────────────────────────
print("\n" + "="*50)
print("SWEEP 4: Input Field")
print("="*50)

for campo in ["title", "text", "title+text"]:
    if campo == "title+text":  # ya lo tenemos
        continue
    config = {"lr": 1e-3, "num_filters": 100, "dropout": 0.5, "input_field": campo}
    result = run_experiment(config, f"field_{campo}")
    all_results.append(result)

print("\n✅ Sweeps terminados")


SWEEP 1: Learning Rate
DataLoaders creados:
  Entrenamiento: 50,488 muestras -> 789 batches
  Validacion:    6,311 muestras -> 99 batches
  Prueba:        6,311 muestras -> 99 batches
  Batch size: 64 | Max len: 200
Epoch 01/10 | Train Loss: 0.1534  Train F1: 0.9347 | Val Loss: 0.0988  Val F1: 0.9615
Epoch 02/10 | Train Loss: 0.0899  Train F1: 0.9709 | Val Loss: 0.1515  Val F1: 0.9588
Epoch 03/10 | Train Loss: 0.0884  Train F1: 0.9778 | Val Loss: 0.2125  Val F1: 0.9595
Epoch 04/10 | Train Loss: 0.0818  Train F1: 0.9825 | Val Loss: 0.2931  Val F1: 0.9621
Epoch 05/10 | Train Loss: 0.0915  Train F1: 0.9843 | Val Loss: 0.3726  Val F1: 0.9634
Epoch 06/10 | Train Loss: 0.1023  Train F1: 0.9864 | Val Loss: 0.6681  Val F1: 0.9564
Epoch 07/10 | Train Loss: 0.0799  Train F1: 0.9903 | Val Loss: 0.7617  Val F1: 0.9591
Epoch 08/10 | Train Loss: 0.1246  Train F1: 0.9902 | Val Loss: 1.0438  Val F1: 0.9584

Early stopping en epoca 8. Mejor Val F1: 0.9634

[lr_0.01] val_f1=0.9634 | test_f1=0.9623
Data

In [4]:
import os

# ── Tabla resumen ────────────────────────────────────────────────────────
# Agregar manualmente lr_0.001 (config base que ya teníamos del sweep)
all_results.append({
    "experiment_id": "lr_0.001",
    "lr": 1e-3, "num_filters": 100, "dropout": 0.5, "input_field": "title+text",
    "val_f1": 0.9745, "val_accuracy": None,
    "test_f1": 0.9731, "test_accuracy": None
})

df_results = pd.DataFrame(all_results)
df_results = df_results.sort_values("val_f1", ascending=False).reset_index(drop=True)

print(df_results[["experiment_id", "lr", "num_filters", "dropout",
                   "input_field", "val_f1", "test_f1"]].to_string(index=False))

# Guardar CSV
os.makedirs("../results", exist_ok=True)
df_results.to_csv("../results/experiments.csv", index=False)
print("\n✅ Guardado en ../results/experiments.csv")

# ── Mejor configuración ──────────────────────────────────────────────────
mejor = df_results.iloc[0]
print(f"\n🏆 MEJOR CONFIG por val_f1:")
print(f"   Experimento: {mejor['experiment_id']}")
print(f"   lr={mejor['lr']} | num_filters={mejor['num_filters']} | dropout={mejor['dropout']}")
print(f"   input_field={mejor['input_field']}")
print(f"   val_f1={mejor['val_f1']} | test_f1={mejor['test_f1']}")

experiment_id     lr  num_filters  dropout input_field  val_f1  test_f1
       dp_0.3 0.0010          100      0.3  title+text  0.9752   0.9771
     lr_0.001 0.0010          100      0.5  title+text  0.9745   0.9731
       nf_200 0.0010          200      0.5  title+text  0.9745   0.9741
     lr_0.001 0.0010          100      0.5  title+text  0.9745   0.9731
       dp_0.7 0.0010          100      0.7  title+text  0.9723   0.9739
        nf_50 0.0010           50      0.5  title+text  0.9706   0.9736
   field_text 0.0010          100      0.5        text  0.9675   0.9669
      lr_0.01 0.0100          100      0.5  title+text  0.9634   0.9623
    lr_0.0001 0.0001          100      0.5  title+text  0.9578   0.9604
  field_title 0.0010          100      0.5       title  0.8923   0.8989

✅ Guardado en ../results/experiments.csv

🏆 MEJOR CONFIG por val_f1:
   Experimento: dp_0.3
   lr=0.001 | num_filters=100 | dropout=0.3
   input_field=title+text
   val_f1=0.9752 | test_f1=0.9771


In [5]:
# ── Reentrenar mejor modelo para analizar errores ────────────────────────
torch.manual_seed(42)

best_config = {
    "lr": mejor["lr"],
    "num_filters": int(mejor["num_filters"]),
    "dropout": mejor["dropout"],
    "input_field": mejor["input_field"]
}

X_tr  = preparar_textos(train_df, best_config["input_field"])
X_val = preparar_textos(val_df,   best_config["input_field"])
X_te  = preparar_textos(test_df,  best_config["input_field"])

train_loader_b, val_loader_b, test_loader_b = create_dataloaders(
    train_data=(X_tr,  labels_train),
    val_data  =(X_val, labels_val),
    test_data =(X_te,  labels_test),
    word2idx=word2idx, batch_size=64, max_len=200
)

best_model = TextCNN(
    vocab_size=len(word2idx),
    embed_dim=50,
    num_filters=best_config["num_filters"],
    kernel_sizes=[3, 4, 5],
    dropout=best_config["dropout"],
    pretrained_embeddings=embedding_matrix
).to(device)

optimizer_best = torch.optim.Adam(best_model.parameters(), lr=best_config["lr"])
criterion = nn.BCEWithLogitsLoss()

history_best = train_model(best_model, train_loader_b, val_loader_b,
                            optimizer_best, criterion, device, num_epochs=10, patience=3)

# ── 10 ejemplos mal clasificados ─────────────────────────────────────────
from src.evaluate import get_predictions

y_true, y_pred, y_proba = get_predictions(best_model, test_loader_b, device)

# Encontrar errores
errores_idx = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]][:10]

print(f"\n{'='*60}")
print("10 EJEMPLOS MAL CLASIFICADOS")
print(f"{'='*60}")

textos_test = preparar_textos(test_df, best_config["input_field"])
test_df_reset = test_df.reset_index(drop=True)

for i in errores_idx:
    print(f"\n[{i}] Real: {'Fake' if y_true[i]==1 else 'Real'} | "
          f"Predicho: {'Fake' if y_pred[i]==1 else 'Real'} | "
          f"Prob: {y_proba[i]:.3f}")
    print(f"Texto: {textos_test[i][:150]}...")

DataLoaders creados:
  Entrenamiento: 50,488 muestras -> 789 batches
  Validacion:    6,311 muestras -> 99 batches
  Prueba:        6,311 muestras -> 99 batches
  Batch size: 64 | Max len: 200
Epoch 01/10 | Train Loss: 0.1794  Train F1: 0.9144 | Val Loss: 0.0926  Val F1: 0.9580
Epoch 02/10 | Train Loss: 0.0684  Train F1: 0.9720 | Val Loss: 0.0779  Val F1: 0.9656
Epoch 03/10 | Train Loss: 0.0381  Train F1: 0.9855 | Val Loss: 0.0647  Val F1: 0.9733
Epoch 04/10 | Train Loss: 0.0184  Train F1: 0.9939 | Val Loss: 0.0720  Val F1: 0.9730
Epoch 05/10 | Train Loss: 0.0104  Train F1: 0.9968 | Val Loss: 0.0708  Val F1: 0.9748
Epoch 06/10 | Train Loss: 0.0051  Train F1: 0.9986 | Val Loss: 0.0778  Val F1: 0.9752
Epoch 07/10 | Train Loss: 0.0038  Train F1: 0.9992 | Val Loss: 0.0912  Val F1: 0.9718
Epoch 08/10 | Train Loss: 0.0033  Train F1: 0.9991 | Val Loss: 0.0941  Val F1: 0.9718
Epoch 09/10 | Train Loss: 0.0023  Train F1: 0.9994 | Val Loss: 0.0950  Val F1: 0.9742

Early stopping en epoca 9. Mejor

## Reflexión: ¿Qué hiperparámetro tiene mayor impacto?

El hiperparámetro con mayor impacto fue el **INPUT FIELD**, seguido del **learning rate**.

### Evidencia con datos

**Sweep Input Field:**

| Experimento | val_f1 | test_f1 |
|---|---|---|
| title+text | 0.9752 | 0.9771 |
| text | 0.9675 | 0.9669 |
| title | 0.8923 | 0.8989 |

Diferencia entre *title* y *title+text*: **0.083 en val_f1**

**Sweep Dropout (para comparar):**

| dropout | val_f1 |
|---|---|
| 0.3 | 0.9752 |
| 0.5 | 0.9745 |
| 0.7 | 0.9723 |

Diferencia máxima: **0.003** — impacto mínimo.

### ¿Por qué?

El campo de entrada determina cuánta información semántica recibe el modelo.
El título solo tiene ~10 palabras promedio, insuficiente para capturar el contexto
completo de una noticia. Combinar **título+texto** da al modelo acceso a toda la
información disponible, lo que mejora significativamente la clasificación.

El dropout y num_filters afectan la capacidad de regularización y representación,
pero la señal del texto ya es tan rica que estos ajustes tienen impacto marginal
en este dataset.

### Mejor configuración encontrada

| Parámetro | Valor |
|---|---|
| lr | 0.001 |
| num_filters | 100 |
| dropout | 0.3 |
| input_field | title+text |
| **val_f1** | **0.9752** |
| **test_f1** | **0.9771** |